In [1]:
install.packages("RSQLite")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
library(RSQLite)
conn <- dbConnect(RSQLite::SQLite(), "crop_data.db")

In [4]:
# 1. Download the datasets from the URLs into R dataframes
crop_data <- read.csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Annual_Crop_Data.csv")
farm_prices <- read.csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_Farm_Prices.csv")
daily_fx <- read.csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Daily_FX.csv")
monthly_fx <- read.csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_FX.csv")

# 2. Load the dataframes into your database tables
dbWriteTable(conn, "CROP_DATA", crop_data, overwrite = TRUE)
dbWriteTable(conn, "FARM_PRICES", farm_prices, overwrite = TRUE)
dbWriteTable(conn, "DAILY_FX", daily_fx, overwrite = TRUE)
dbWriteTable(conn, "MONTHLY_FX", monthly_fx, overwrite = TRUE)

# 3. Verify the tables exist in the database
dbListTables(conn)

[1] "CROP_DATA"   "DAILY_FX"    "FARM_PRICES" "MONTHLY_FX"

In [5]:
# Problem 3: How many records are in the farm prices dataset?
# We use the COUNT(*) function to calculate the total number of rows in the table.
dbGetQuery(conn, "SELECT COUNT(*) FROM FARM_PRICES")

COUNT(*)
<int>
2678


In [6]:
# Problem 4: Which provinces are included in the farm prices dataset?
# The DISTINCT keyword ensures each province (GEO column) is only listed once.
dbGetQuery(conn, "SELECT DISTINCT GEO FROM FARM_PRICES")

GEO
<chr>
Alberta
Saskatchewan


In [9]:
# Problem 5: How many hectares of Rye were harvested in Canada in 1968?
dbGetQuery(conn, "SELECT HARVESTED_AREA FROM CROP_DATA WHERE CROP_TYPE = 'Rye' AND YEAR = 1968 AND GEO = 'Canada'")

HARVESTED_AREA
<int>


In [10]:
# Problem 6: Query and display the first 6 rows of the farm prices table for Rye.
dbGetQuery(conn, "SELECT * FROM FARM_PRICES WHERE CROP_TYPE = 'Rye' LIMIT 6")

CD_ID,DATE,CROP_TYPE,GEO,PRICE_PRERMT
<int>,<chr>,<chr>,<chr>,<dbl>
4,1985-01-01,Rye,Alberta,100.77
5,1985-01-01,Rye,Saskatchewan,109.75
10,1985-02-01,Rye,Alberta,95.05
11,1985-02-01,Rye,Saskatchewan,103.46
16,1985-03-01,Rye,Alberta,96.77
17,1985-03-01,Rye,Saskatchewan,106.38


In [11]:
# Problem 7: Which provinces grew Barley?
dbGetQuery(conn, "SELECT DISTINCT GEO FROM CROP_DATA WHERE CROP_TYPE = 'Barley'")

GEO
<chr>
Alberta
Canada
Saskatchewan


In [14]:
# Problem 8: Find the first and last dates for the farm prices data.
dbGetQuery(conn, "SELECT MIN(DATE), MAX(DATE) FROM FARM_PRICES")

MIN(DATE),MAX(DATE)
<chr>,<chr>
1985-01-01,2020-12-01


In [15]:
# Problem 9: Which crops have ever reached a farm price greater than or equal to $350 per metric tonne?
dbGetQuery(conn, "SELECT DISTINCT CROP_TYPE FROM FARM_PRICES WHERE PRICE_PRERMT >= 350")

CROP_TYPE
<chr>
Canola


In [16]:
# Problem 10: Rank the crop types harvested in Saskatchewan in the year 2000 by their average yield. Which crop performed best?
dbGetQuery(conn, "SELECT CROP_TYPE, AVG_YIELD FROM CROP_DATA WHERE GEO = 'Saskatchewan' AND YEAR = 2000 ORDER BY AVG_YIELD DESC")

CROP_TYPE,AVG_YIELD
<chr>,<int>


In [17]:
# Problem 11: Rank the crops and geographies by their average yield (KG per hectare) since the year 2000.
dbGetQuery(conn, "SELECT GEO, CROP_TYPE, AVG_YIELD FROM CROP_DATA WHERE YEAR >= 2000 ORDER BY AVG_YIELD DESC")

GEO,CROP_TYPE,AVG_YIELD
<chr>,<chr>,<int>
Alberta,Barley,4100
Alberta,Barley,4100
Alberta,Barley,3980
Alberta,Wheat,3900
Canada,Barley,3900
Alberta,Wheat,3900
Alberta,Barley,3900
Alberta,Barley,3890
Canada,Barley,3820


In [18]:
# Problem 12: Use a subquery to determine how much wheat was harvested in Canada in the most recent year of the data.
dbGetQuery(conn, "SELECT HARVESTED_AREA FROM CROP_DATA WHERE CROP_TYPE = 'Wheat' AND GEO = 'Canada' AND YEAR = (SELECT MAX(YEAR) FROM CROP_DATA)")

HARVESTED_AREA
<int>
10017800


In [19]:
# Problem 13: Calculate the monthly price of Saskatchewan Canola in CAD and USD for the most recent 6 months.
dbGetQuery(conn, "
  SELECT
    F.DATE,
    F.PRICE_PRERMT AS Price_CAD,
    (F.PRICE_PRERMT / M.FXUSDCAD) AS Price_USD
  FROM
    FARM_PRICES F,
    MONTHLY_FX M
  WHERE
    F.DATE = M.DATE
    AND F.CROP_TYPE = 'Canola'
    AND F.GEO = 'Saskatchewan'
  ORDER BY
    F.DATE DESC
  LIMIT 6
")

DATE,Price_CAD,Price_USD
<chr>,<dbl>,<dbl>
2020-12-01,507.33,396.1128
2020-11-01,495.64,379.2718
2020-10-01,474.80,359.2965
2020-09-01,463.52,350.4057
2020-08-01,464.60,351.3827
2020-07-01,462.88,342.9122
